In [ ]:
# Cell 1 — pull and merge Akola Jassid + Aphid, 2005-2009
import pandas as pd, requests, time
from io import StringIO

def pull(pest):
    url = "http://icar-crida.res.in:8080/naip/Servlet1.jsp"
    params = {"combo0": "Cotton", "combo1": "Akola", "combo2": pest,
              "combo3": "2005", "combo4": "2009"}
    resp = requests.get(url, params=params, timeout=15)
    return pd.read_html(StringIO(resp.text))[2]

jassid = pull("Jassid").rename(columns={"Pest Value": "Jassid_Value"})
time.sleep(1)
aphid = pull("Aphid").rename(columns={"Pest Value": "Aphid_Value"})

df = jassid.merge(
    aphid[["Observation Year", "Standard Week", "Aphid_Value"]],
    on=["Observation Year", "Standard Week"], how="outer"
).sort_values(["Observation Year", "Standard Week"]).reset_index(drop=True)
print(df.shape, df.columns.tolist())

In [ ]:
# Cell 2 — impute within-season, normalize, combine into one pest_pressure target
for col in ["Jassid_Value", "Aphid_Value"]:
    df[col] = df.groupby("Observation Year")[col].transform(lambda x: x.interpolate())
df = df.dropna(subset=["Jassid_Value", "Aphid_Value"]).reset_index(drop=True)

# min-max scale each pest column using ONLY train years (2005-2008) to avoid leakage
train_mask = df["Observation Year"] <= 2008
for col in ["Jassid_Value", "Aphid_Value"]:
    lo, hi = df.loc[train_mask, col].min(), df.loc[train_mask, col].max()
    df[col + "_norm"] = ((df[col] - lo) / (hi - lo)).clip(0, 1)

df["pest_pressure"] = df[["Jassid_Value_norm", "Aphid_Value_norm"]].max(axis=1)
print(df[["Observation Year", "Standard Week", "pest_pressure"]].head())

In [ ]:
# Cell 3 — build sliding windows (4-week lookback) per season, split by year
import numpy as np

weather_cols = ["MaxT(°C)", "MinT(°C)", "RH1(%)", "RH2(%)", "RF(mm)", "WS(kmph)", "SSH(hrs)", "EVP(mm)"]
feature_cols = weather_cols + ["pest_pressure"]
WINDOW = 4

def make_sequences(sub_df):
    X, y = [], []
    vals = sub_df[feature_cols].values
    target = sub_df["pest_pressure"].values
    for i in range(WINDOW, len(vals)):
        X.append(vals[i-WINDOW:i])
        y.append(target[i])
    return np.array(X), np.array(y)

train_df = df[df["Observation Year"] <= 2008]
test_df  = df[df["Observation Year"] == 2009]

X_train, y_train = make_sequences(train_df)
X_test,  y_test  = make_sequences(test_df)
print(X_train.shape, X_test.shape)

In [ ]:
# Cell 4 — small LSTM, regression head
from tensorflow.keras import layers, models, callbacks

model = models.Sequential([
    layers.Input(shape=(WINDOW, len(feature_cols))),
    layers.LSTM(16, dropout=0.3),
    layers.Dense(1, activation="sigmoid")
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

es = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                     epochs=50, batch_size=8, callbacks=[es], verbose=1)

In [ ]:
# Cell 5 — evaluate
preds = model.predict(X_test).flatten()
mae = np.mean(np.abs(preds - y_test))
print(f"Test MAE: {mae:.4f}")
print("Sample predictions vs actual:", list(zip(preds[:10].round(3), y_test[:10].round(3))))

In [ ]:
# Cell 6 — does the LSTM beat trivial guessing?
mean_baseline = np.full_like(y_test, y_train.mean())
persistence_baseline = X_test[:, -1, -1]  # last known pest_pressure value per window

print("LSTM MAE:         ", mae)
print("Mean-baseline MAE: ", np.mean(np.abs(mean_baseline - y_test)))
print("Persistence MAE:   ", np.mean(np.abs(persistence_baseline - y_test)))